In [2]:
%pip install pyarrow



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import meteostat as ms
from meteostat import Point, daily
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import sqlite3
from datetime import date


In [4]:
# 1) Download the stations database (once)
STATIONS_DB_URL = "https://data.meteostat.net/stations.db"
STATIONS_DB_FILE = "stations.db"

with requests.get(STATIONS_DB_URL, stream=True) as r:
    r.raise_for_status()
    with open(STATIONS_DB_FILE, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# 2) Open the DB and get all station IDs in France
conn = sqlite3.connect(STATIONS_DB_FILE)
cur = conn.cursor()

# country code 'FR' for France
cur.execute("SELECT id,region,latitude, longitude FROM stations WHERE country = 'FR'")
rows = cur.fetchall()
conn.close()

# Opret en Pandas DataFrame direkte fra resultatet for nem håndtering
# Kolonnerne vi skal bruge er 'id', 'latitude' og 'longitude'
df_stations = pd.DataFrame(rows, columns=['station_id', 'region','lat', 'lon'])

print(f"DataFrame created with {len(df_stations)} stations and coordinates.")
print(df_stations.head())



DataFrame created with 223 stations and coordinates.
  station_id region      lat     lon
0      07666      K  42.9183  3.0617
1      07690      U  43.6500  7.2000
2      07003      O  50.5167  1.6167
3      LFYL0      I  47.7000  6.5500
4      07222      R  47.1667 -1.6000


### Using KNN

In [5]:

def _normalize_dep_code(x) -> str:
    """
    Normaliser franske departementkoder så de matcher på tværs af kilder:
    - 1..9  -> "01".."09"
    - 10..95 -> "10".."95"
    - 971..976 beholdes (3 cifre)
    - 2A/2B beholdes
    - fjerner evt. '.0'
    """
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if s.endswith(".0"):
        s = s[:-2]
    # Behold 2A/2B osv.
    if any(ch.isalpha() for ch in s):
        return s
    # Kun tal
    if s.isdigit():
        if len(s) == 1:
            return s.zfill(2)
        return s  # 2+ cifre beholdes (inkl 971 osv)
    return s


def build_dep_station_weights_idw(
    df_stations: pd.DataFrame,
    gdf_departements: gpd.GeoDataFrame,
    dep_code_col: str = "code",
    station_id_col: str = "station_id",
    lon_col: str = "lon",
    lat_col: str = "lat",
    K: int = 5,
    p: float = 2.0,
    crs_stations: str = "EPSG:4326",
    crs_distance: str = "EPSG:2154",  # Lambert-93 (meter)
) -> pd.DataFrame:
    """
    Fast (tid-invariant) mapping departement -> top-K stationer med IDW-vægte.
    Returnerer: ['departement_code','station_id','distance_m','weight']
    """

    # --- 1) Stations GeoDataFrame ---
    req = {station_id_col, lon_col, lat_col}
    missing = req - set(df_stations.columns)
    if missing:
        raise ValueError(f"df_stations mangler kolonner: {missing}")

    df_st = df_stations[[station_id_col, lon_col, lat_col]].copy()
    df_st = df_st.dropna(subset=[station_id_col, lon_col, lat_col])

    gdf_st = gpd.GeoDataFrame(
        df_st,
        geometry=gpd.points_from_xy(df_st[lon_col], df_st[lat_col]),
        crs=crs_stations
    )

    # --- 2) Departementer: kode + repræsentativt punkt (altid inde i polygon) ---
    if dep_code_col not in gdf_departements.columns:
        raise ValueError(
            f"dep_code_col='{dep_code_col}' findes ikke i gdf_departements. "
            f"Tilgængelige kolonner: {list(gdf_departements.columns)}"
        )

    gdf_dep = gdf_departements[[dep_code_col, "geometry"]].copy()

    # normaliser dep-koder så merges senere ikke fejler pga '1' vs '01'
    gdf_dep["departement_code"] = gdf_dep[dep_code_col].apply(_normalize_dep_code)

    # projectér til meter for korrekt afstand
    gdf_dep = gdf_dep.to_crs(crs_distance)
    gdf_st = gdf_st.to_crs(crs_distance)

    # brug representative point (sikrere end centroid)
    gdf_dep["rep_point"] = gdf_dep.geometry.representative_point()

    # --- 3) Krydsprodukt (dep x station) og afstand ---
    dep_tbl = gdf_dep[["departement_code", "rep_point"]].copy()
    st_tbl = gdf_st[[station_id_col, "geometry"]].rename(columns={station_id_col: "station_id"}).copy()

    dep_tbl["key"] = 1
    st_tbl["key"] = 1
    pairs = dep_tbl.merge(st_tbl, on="key").drop(columns=["key"])

    pairs["distance_m"] = pairs["rep_point"].distance(pairs["geometry"])

    # --- 4) Vælg top-K nærmeste stationer pr departement ---
    pairs = pairs.sort_values(["departement_code", "distance_m"])
    topk = pairs.groupby("departement_code", as_index=False).head(K).copy()

    # --- 5) IDW weights ---
    eps = 1e-9
    topk["distance_m_safe"] = topk["distance_m"].clip(lower=eps)
    topk["idw"] = 1.0 / np.power(topk["distance_m_safe"], p)

    def normalize_group(g: pd.DataFrame) -> pd.DataFrame:
        # hvis station ligger præcis på rep_point: giv den fuld vægt
        if (g["distance_m"] <= 1e-6).any():
            z = g["distance_m"] <= 1e-6
            g = g.copy()
            g["weight"] = 0.0
            g.loc[z, "weight"] = 1.0 / z.sum()
            return g
        s = g["idw"].sum()
        g = g.copy()
        g["weight"] = g["idw"] / s if s > 0 else 1.0 / len(g)
        return g

    dep_station_weights = (
        topk.groupby("departement_code", group_keys=False)
            .apply(normalize_group)
            .loc[:, ["departement_code", "station_id", "distance_m", "weight"]]
            .reset_index(drop=True)
    )

    # sanity checks
    chk = dep_station_weights.groupby("departement_code")["weight"].sum()
    if not np.allclose(chk.values, 1.0, atol=1e-8):
        bad = chk[~np.isclose(chk.values, 1.0, atol=1e-8)]
        raise RuntimeError(f"Vægte summer ikke til 1 for disse departementer:\n{bad}")

    # check at hvert dep har K rækker (kan fejle hvis du har <K stationer totalt)
    counts = dep_station_weights.groupby("departement_code")["station_id"].count()
    if (counts < min(K, dep_station_weights["station_id"].nunique())).any():
        print("ADVARSEL: Nogle departementer har færre end K stationer (typisk fordi station-universet er lille).")

    return dep_station_weights


def apply_dep_weights_to_station_series(
    df_station_series: pd.DataFrame,
    dep_station_weights: pd.DataFrame,
    value_col: str,
    date_col: str = "date"
) -> pd.DataFrame:
    """
    Input: df_station_series med kolonner [date_col, 'station_id', value_col]
    Output: departement-serie [date_col, 'departement_code', value_col]
    """
    need = {date_col, "station_id", value_col}
    missing = need - set(df_station_series.columns)
    if missing:
        raise ValueError(f"df_station_series mangler kolonner: {missing}")

    x = df_station_series.merge(dep_station_weights, on="station_id", how="inner")
    x["weighted_value"] = x[value_col] * x["weight"]

    dep_series = (
        x.groupby([date_col, "departement_code"], as_index=False)["weighted_value"]
         .sum()
         .rename(columns={"weighted_value": value_col})
    )
    return dep_series


# -------------------------
# EKSEMPEL PÅ BRUG
# -------------------------
gdf_departements = gpd.read_file("Meta/departements-1000m.geojson")

dep_station_weights = build_dep_station_weights_idw(
    df_stations=df_stations,
    gdf_departements=gdf_departements,
    dep_code_col="code",   # ret hvis din geojson bruger et andet navn
    K=5,
    p=2.0
)

print(dep_station_weights.head())
dep_station_weights.to_csv("dep_station_KNN_weights.csv", index=False)
print("Saved: dep_station_KNN_weights.csv")


  departement_code station_id    distance_m    weight
0               01      07482  10397.027129  0.837711
1               01      07481  40405.144269  0.055468
2               01      07385  45102.814698  0.044515
3               01      07480  46890.005665  0.041186
4               01      LFLP0  65478.895034  0.021121
Saved: dep_station_KNN_weights.csv


/var/folders/g6/v5w89nm96b732r3v5w91v8dm0000gn/T/ipykernel_79425/3841705967.py:111: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(normalize_group)


## using regions


In [7]:
gdf_departements = gpd.read_file("Meta/regions-1000m.geojson")

dep_station_weights = build_dep_station_weights_idw(
    df_stations=df_stations,
    gdf_departements=gdf_departements,
    dep_code_col="code",   # ret hvis din geojson bruger et andet navn
    K=5,
    p=2.0
)

print(dep_station_weights.head())
dep_station_weights.to_csv("reg_station_KNN_weights.csv", index=False)
print("Saved: reg_station_KNN_weights.csv")


TypeError: build_dep_station_weights_idw() got an unexpected keyword argument 'gdf_regions'

### Using mean within department

import pandas as pd
import meteostat as ms
from datetime import date
import time, random

# -------------------------
# 1) Load station IDs with a department
# -------------------------
station_to_dep = pd.read_csv("station_to_dep.csv")
station_to_dep = station_to_dep.dropna(subset=["departement_code"]).copy()
station_to_dep["station_id"] = station_to_dep["station_id"].astype(str)

station_ids = station_to_dep["station_id"].unique().tolist()
print(f"Stations with department: {len(station_ids)}")
print("Sample:", station_ids[:5])

# -------------------------
# 2) Fetch daily temperatures (2000–2024) using ms.daily(...)
# -------------------------
start = date(2000, 1, 1)
end   = date(2024, 12, 31)

all_frames = []
ok = empty = fail = 0

def fetch_station_daily(sid: str, retries: int = 4, base_sleep: float = 0.8):
    last = None
    for a in range(retries):
        try:
            st = ms.Station(id=sid)
            df = ms.daily(st, start, end).fetch()
            return df
        except Exception as e:
            last = e
            time.sleep(base_sleep * (2 ** a) + random.random())
    raise last

for i, sid in enumerate(station_ids, start=1):
    try:
        df = fetch_station_daily(sid)

        if df is None or df.empty:
            empty += 1
            continue

        df = df.reset_index()  # brings time out as column (often named 'time')

        # Normalize time column name
        if "time" in df.columns:
            df = df.rename(columns={"time": "date"})
        elif "date" not in df.columns:
            # if index column got a different name, assume first col is date-like
            df = df.rename(columns={df.columns[0]: "date"})

        # Build temp column: prefer tavg, else compute from tmin/tmax
        if "tavg" in df.columns and df["tavg"].notna().any():
            out = df[["date", "tavg"]].rename(columns={"tavg": "temp"})
        elif "tmin" in df.columns and "tmax" in df.columns:
            out = df[["date", "tmin", "tmax"]].copy()
            out["temp"] = (out["tmin"] + out["tmax"]) / 2.0
            out = out[["date", "temp"]]
        else:
            empty += 1
            continue

        out["station_id"] = sid
        all_frames.append(out[["date", "station_id", "temp"]])
        ok += 1

    except Exception as e:
        fail += 1
        print(f"[WARN] {sid} failed: {type(e).__name__}: {e}")

    if i % 25 == 0 or i == len(station_ids):
        print(f"{i}/{len(station_ids)} | ok={ok} empty={empty} fail={fail} frames={len(all_frames)}")

# -------------------------
# 3) Combine + save (safe)
# -------------------------
if len(all_frames) == 0:
    raise RuntimeError("No objects to concatenate: no station returned usable data (tavg or tmin/tmax).")

df_temp = pd.concat(all_frames, ignore_index=True)

print(df_temp.head())
print("df_temp shape:", df_temp.shape)
print("unique stations fetched:", df_temp["station_id"].nunique())

df_temp.to_parquet("france_daily_temp_2000_2024.parquet", index=False)
print("Saved: france_daily_temp_2000_2024.parquet")


In [ ]:
## here calculate the departement-level mean 